# Warps in the model, and paced latency

Two follow-ups to the first T4 run (`docs/megakernel_sync.md`, parts 3 and 4).

1. **Warp-level mutants with oracle labels.** The step machine now models warps (`no_barrier`), so
   `races --backend cuda` labels those mutants with the oracle instead of `unmodelled`. The question:
   are the mutants the GPU shows exactly the ones the oracle calls `racy`?
2. **Paced latency.** Every back-to-back latency series failed tinytail's i.i.d. gates. This reruns
   them with `--design paced`: rounds of every schedule in a random order, each launch after a random
   idle gap. Do the gates pass now, and do the paired per-round differences separate the schedules?

In [ ]:
# Kaggle: GPU T4, internet on, and the private dataset rajulkabir/tileproof-src attached
# (src/triton_equiv as a tarball, with SOURCE.txt naming the commit it was made from).
import glob, os, shutil, subprocess, sys, tarfile
os.environ["TRITON_INTERPRET"] = "0"
SRC = "/kaggle/working/tileproof_src"
dirs = glob.glob("/kaggle/input/**/triton_equiv", recursive=True)
tars = glob.glob("/kaggle/input/**/tileproof_src.tar.gz", recursive=True)
if dirs:
    shutil.copytree(dirs[0].rsplit("/", 1)[0], SRC, dirs_exist_ok=True)
elif tars:
    os.makedirs(SRC, exist_ok=True)
    with tarfile.open(tars[0]) as t:
        t.extractall(SRC)
else:
    raise SystemExit("attach the tileproof-src dataset: this notebook needs code newer than GitHub's main")
print(open(SRC + "/SOURCE.txt").read())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "z3-solver", "ninja",
                "git+https://github.com/rajul-kk/tinytail.git"], check=True)
sys.path[:0] = [SRC]
# tinytail before 0.0.2 imports InterpolationWarning where newer statsmodels no longer has it
import statsmodels.tsa.stattools as _st
if not hasattr(_st, "InterpolationWarning"):
    from statsmodels.tools.sm_exceptions import InterpolationWarning as _iw
    _st.InterpolationWarning = _iw

In [ ]:
import json, torch
print(torch.__version__, torch.cuda.get_device_name(0))
from triton_equiv.cuda_ref import Backend
from triton_equiv.msync import cuda_vm, latency, races, warp_check
print(cuda_vm.build_error(Backend.CUDA) or "CUDA VM built")

## 1. Race counts with the warp mutants labelled by the oracle

In [ ]:
res = races.main(["--backend", "cuda", "--reps", "100", "--out", "/kaggle/working/msync_gpu_races_cuda_warps.json"])
print(json.dumps(res["warp"], indent=1))
for r in warp_check.compare(res["rows"]):
    print(r["id"], r["label"], r["rate_random"], "agrees" if r["agrees"] else "DISAGREES")

## 2. Paced latency, Triton and CUDA

In [ ]:
lat_p = latency.main(["--design", "paced", "--n", "100000", "--out", "/kaggle/working/msync_gpu_latency_paced.json"])
print(json.dumps(lat_p["paired"], indent=1))

In [ ]:
lat_pc = latency.main(["--backend", "cuda", "--design", "paced", "--n", "100000",
                       "--out", "/kaggle/working/msync_gpu_latency_paced_cuda.json"])
print(json.dumps(lat_pc["paired"], indent=1))
for r in lat_p["rows"] + lat_pc["rows"]:
    print(r["program"], r["schedule"], round(r["median_us"], 2), r["gate_verdict"], r["report"]["gates"]["failed"])